# 02 · Locked Audio Teacher — USSD ComParE16 + LSTM

Reproduces the accepted Ravi et al. USSD released run #4 checkpoint.

- checkpoint: `md_35_epochs.pth`
- author commit: `c3e68649153004ed2174878a1c54c716ad26cfd7`
- DEV: 34 participants, 440 excluded
- TRAIN target view: author-style run-4 seed 1300 crop
- no fine-tuning
- TEST is never opened


In [ ]:
from google.colab import drive,userdata
drive.mount('/content/drive')

from pathlib import Path
import base64,subprocess,sys,json
REPO='engrkhubabahmad/reliability-aware-depression-kd'
ROOT=Path('/content/reliability-aware-depression-kd')
token=userdata.get('GITHUB_TOKEN'); assert token,'Add GITHUB_TOKEN to Colab Secrets'
auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git=['git','-c',f'http.extraHeader=AUTHORIZATION: basic {auth}']
if not (ROOT/'.git').exists():
    subprocess.run(git+['clone',f'https://github.com/{REPO}.git',str(ROOT)],check=True)
subprocess.run(git+['-C',str(ROOT),'fetch','origin','main'],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','main'],check=True)
subprocess.run(git+['-C',str(ROOT),'pull','--ff-only','origin','main'],check=True)
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements-colab.txt')])
print('main:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())


## Pin author source and locate openSMILE 3.0.2


In [ ]:
AUTHOR=Path('/content/drive/MyDrive/tools/solo_teacher_sources/USSD-depression')
subprocess.run([
 sys.executable,'-m','scripts.teachers.ussd_audio.bootstrap',
 '--root',str(AUTHOR)
],cwd=ROOT,check=True)

SMILE_DRIVE=Path('/content/drive/MyDrive/tools/opensmile-3.0.2-linux-x86_64')
SMILE_LOCAL=Path('/content/opensmile-3.0.2-linux-x86_64')
assert SMILE_DRIVE.exists(),'Missing Drive openSMILE copy'
if not SMILE_LOCAL.exists():
    subprocess.run(['cp','-r',str(SMILE_DRIVE),'/content/'],check=True)
SMILExtract=SMILE_LOCAL/'bin/SMILExtract'
COMPARE16=SMILE_LOCAL/'config/compare16/ComParE_2016.conf'
subprocess.run(['chmod','+x',str(SMILExtract)],check=True)
assert SMILExtract.exists() and COMPARE16.exists()
print(SMILExtract)
print(COMPARE16)


## Prepare/audit DEV-34

Cached participants are reused. This stage performs no fitting.


In [ ]:
FEAT=Path('/content/drive/MyDrive/DAIC_WOZ/experiments/ussd_compare16')
AUDIT=Path('/content/drive/MyDrive/DAIC_WOZ/experiments/ussd_frozen_audit')
subprocess.run([
 sys.executable,'-m','scripts.teachers.ussd_audio.prepare_compare16',
 '--daic-root','/content/drive/MyDrive/DAIC_WOZ','--output',str(FEAT),
 '--split','dev','--smile-extract',str(SMILExtract),'--compare16-config',str(COMPARE16)
],cwd=ROOT,check=True)
subprocess.run([
 sys.executable,'-m','scripts.teachers.ussd_audio.audit_frozen',
 '--features',str(FEAT),'--author-root',str(AUTHOR),'--output',str(AUDIT)
],cwd=ROOT,check=True)


## Prepare TRAIN-107 and export author-style run-4 KD targets


In [ ]:
FINAL=Path('/content/drive/MyDrive/DAIC_WOZ/experiments/ussd_audio_teacher_final')
subprocess.run([
 sys.executable,'-m','scripts.teachers.ussd_audio.prepare_compare16',
 '--daic-root','/content/drive/MyDrive/DAIC_WOZ','--output',str(FEAT),
 '--split','train','--smile-extract',str(SMILExtract),'--compare16-config',str(COMPARE16)
],cwd=ROOT,check=True)
subprocess.run([
 sys.executable,'-m','scripts.teachers.ussd_audio.audit_train_crop',
 '--features',str(FEAT),'--author-root',str(AUTHOR),'--output',str(FINAL)
],cwd=ROOT,check=True)


## Integrity check


In [ ]:
import pandas as pd,numpy as np
tr=pd.read_csv(FINAL/'train_run4_crop_kd_targets.csv')
dv=pd.read_csv(AUDIT/'dev_participant_predictions.csv')
assert len(tr)==107 and tr.participant_id.nunique()==107
assert len(dv)==34 and dv.participant_id.nunique()==34 and 440 not in set(dv.participant_id)
assert not set(tr.participant_id)&set(dv.participant_id)
q=1/(1+np.exp(-tr.kd_logit.to_numpy(float)))
assert np.allclose(q,tr.kd_probability.to_numpy(float),atol=1e-6)
ta=json.loads((FINAL/'train_run4_crop_audit.json').read_text())
da=json.loads((AUDIT/'audit.json').read_text())
assert ta['test_opened'] is False and da['test_opened'] is False
print('Frozen DEV-34 macro-F1:',da['local_dev34']['macro_f1'])
print('TRAIN crop participants:',ta['participants'])
print('\nAUDIO TEACHER LOCKED: targets verified; TEST CLOSED.')
